In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import balanced_accuracy_score

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## Vars:

In [3]:
TARGET = 'class'
N_FOLDS = 5
RANDOM_STATE = 42

In [4]:
train_df = pd.read_csv('data/train.csv')
target_col = train_df[TARGET]
train_df=  train_df.drop(columns=['id', TARGET]) # Remove the id and class cols

test_df = pd.read_csv('data/test.csv')

### Feature engineering

In [5]:
for df in [train_df, test_df]:
    df['u_g'] = df['u'] - df['g']
    df['u_r'] = df['u'] - df['r']
    df['u_i'] = df['u'] - df['i']
    df['u_z'] = df['u'] - df['z']

    df['g_r'] = df['g'] - df['r']
    df['g_i'] = df['g'] - df['i']
    df['g_z'] = df['g'] - df['z']

    df['r_i'] = df['r'] - df['i']
    df['r_z'] = df['r'] - df['z']

    df['i_z'] = df['i'] - df['z']

In [6]:
spectral_type_map = {v: i for i, v in enumerate(train_df['spectral_type'].unique())}
galaxy_population_map = {v: i for i, v in enumerate(train_df['galaxy_population'].unique())}

for df in [train_df, test_df]:
    df['spectral_type_enc'] = df['spectral_type'].map(spectral_type_map)
    df['galaxy_population_enc'] = df['galaxy_population'].map(galaxy_population_map)

# Color ratios
for df in [train_df, test_df]:
    df['u_g_r'] = df['u_g'] / (df['g_r'] + 1e-6)
    df['g_r_i'] = df['g_r'] / (df['r_i'] + 1e-6)
    df['r_i_z'] = df['r_i'] / (df['i_z'] + 1e-6)

    # Redshift interactions
    df['redshift_u'] = df['redshift'] * df['u']
    df['redshift_g'] = df['redshift'] * df['g']
    df['redshift_r'] = df['redshift'] * df['r']
    df['redshift_i'] = df['redshift'] * df['i']
    df['redshift_z'] = df['redshift'] * df['z']

    # Magnitude stats
    mag_cols = ['u', 'g', 'r', 'i', 'z']
    df['mag_mean'] = df[mag_cols].mean(axis=1)
    df['mag_std'] = df[mag_cols].std(axis=1)
    df['mag_min'] = df[mag_cols].min(axis=1)
    df['mag_max'] = df[mag_cols].max(axis=1)
    df['mag_range'] = df['mag_max'] - df['mag_min']

    # Angular features
    df['alpha_delta_ratio'] = df['alpha'] / (df['delta'].abs() + 1e-6)
    df['angular_dist'] = np.sqrt(df['alpha']**2 + df['delta']**2)

train_df.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,u_g,u_r,u_i,u_z,g_r,g_i,g_z,r_i,r_z,i_z,spectral_type_enc,galaxy_population_enc,u_g_r,g_r_i,r_i_z,redshift_u,redshift_g,redshift_r,redshift_i,redshift_z,mag_mean,mag_std,mag_min,mag_max,mag_range,alpha_delta_ratio,angular_dist
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,3.576564,5.114196,6.215010,6.851065,1.537632,2.638446,3.274501,1.100813,1.736869,0.636056,0,0,2.326019,1.396814,1.730684,10.417647,8.954895,8.326031,7.875818,7.615682,21.120756,2.731221,18.621057,25.472123,6.851065,8.711119,148.704497
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,1.691447,3.191300,3.552442,3.992076,1.499854,1.860995,2.300629,0.361141,0.800775,0.439634,0,0,1.127740,4.153085,0.821456,3.282502,3.015294,2.778353,2.721302,2.651850,18.293056,1.636652,16.786433,20.778509,3.992076,3.956775,132.012922
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,-0.043925,-0.136637,0.452574,0.477837,-0.092712,0.496499,0.521762,0.589211,0.614474,0.025263,1,1,0.473784,-0.157350,23.321837,59.398574,59.522609,59.784407,58.120611,58.049273,20.885233,0.292103,20.557366,21.171840,0.614474,5.086814,183.233878
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,2.254320,4.287302,4.939397,5.390104,2.032982,2.685077,3.135783,0.652096,1.102802,0.450706,0,0,1.108873,3.117609,1.446828,12.493809,11.285271,10.195392,9.845805,9.604182,19.930831,2.235331,17.914952,23.305056,5.390104,4.649392,230.982447
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,2.231478,3.468709,3.803711,4.086973,1.237231,1.572233,1.855495,0.335002,0.618264,0.283262,0,0,1.803605,3.693199,1.182650,12.061777,10.821608,10.134002,9.947821,9.790395,18.984984,1.676354,17.616185,21.703158,4.086973,7.332741,143.148996


In [7]:
train_df = train_df.drop(columns=['spectral_type', 'galaxy_population'])
test_df = test_df.drop(columns=['spectral_type', 'galaxy_population'])

In [8]:
target_col = target_col.map({'GALAXY': 0, 'QSO': 1, 'STAR': 2})

In [9]:
train_df.head(10)

,alpha,delta,u,g,r,i,z,redshift,u_g,u_r,u_i,u_z,g_r,g_i,g_z,r_i,r_z,i_z,spectral_type_enc,galaxy_population_enc,u_g_r,g_r_i,r_i_z,redshift_u,redshift_g,redshift_r,redshift_i,redshift_z,mag_mean,mag_std,mag_min,mag_max,mag_range,alpha_delta_ratio,angular_dist
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,3.576564,5.114196,6.215010,6.851065,1.537632,2.638446,3.274501,1.100813,1.736869,0.636056,0,0,2.326019,1.396814,1.730684,10.417647,8.954895,8.326031,7.875818,7.615682,21.120756,2.731221,18.621057,25.472123,6.851065,8.711119,148.704497
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,1.691447,3.191300,3.552442,3.992076,1.499854,1.860995,2.300629,0.361141,0.800775,0.439634,0,0,1.127740,4.153085,0.821456,3.282502,3.015294,2.778353,2.721302,2.651850,18.293056,1.636652,16.786433,20.778509,3.992076,3.956775,132.012922
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,-0.043925,-0.136637,0.452574,0.477837,-0.092712,0.496499,0.521762,0.589211,0.614474,0.025263,1,1,0.473784,-0.157350,23.321837,59.398574,59.522609,59.784407,58.120611,58.049273,20.885233,0.292103,20.557366,21.171840,0.614474,5.086814,183.233878
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,2.254320,4.287302,4.939397,5.390104,2.032982,2.685077,3.135783,0.652096,1.102802,0.450706,0,0,1.108873,3.117609,1.446828,12.493809,11.285271,10.195392,9.845805,9.604182,19.930831,2.235331,17.914952,23.305056,5.390104,4.649392,230.982447
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,2.231478,3.468709,3.803711,4.086973,1.237231,1.572233,1.855495,0.335002,0.618264,0.283262,0,0,1.803605,3.693199,1.182650,12.061777,10.821608,10.134002,9.947821,9.790395,18.984984,1.676354,17.616185,21.703158,4.086973,7.332741,143.148996
5,250.727869,31.756548,20.926469,19.693480,18.902361,19.247572,18.508241,0.076299,1.232989,2.024108,1.678897,2.418228,0.791119,0.445908,1.185239,-0.345211,0.394120,0.739331,2,1,1.558536,-2.291704,-0.466923,1.596664,1.502588,1.442227,1.468566,1.412156,19.455625,0.930978,18.508241,20.926469,2.418228,7.895312,252.730968
6,0.752529,-2.936677,22.829195,22.686143,20.583886,19.781338,19.410491,0.575080,0.143052,2.245309,3.047857,3.418704,2.102257,2.904805,3.275652,0.802549,1.173395,0.370847,0,0,0.068047,2.619473,2.164093,13.128606,13.046340,11.837375,11.375845,11.162579,21.058211,1.609108,19.410491,22.829195,3.418704,0.256252,3.031563
7,235.611325,39.626517,22.511467,21.480306,21.765645,21.508658,21.333476,2.159269,1.031161,0.745822,1.002808,1.177991,-0.285339,-0.028353,0.146830,0.256987,0.432169,0.175182,1,1,-3.613821,-1.110323,1.466957,48.608318,46.381764,46.997888,46.442985,46.064719,21.719910,0.469048,21.333476,22.511467,1.177991,5.945799,238.920400
8,355.359230,2.182312,20.396550,20.064767,19.892257,19.836272,19.860081,0.900087,0.331783,0.504293,0.560278,0.536469,0.172510,0.228495,0.204686,0.055985,0.032177,-0.023809,3,1,1.923262,3.081276,-2.351569,18.358669,18.060035,17.904761,17.854370,17.875799,20.009985,0.233956,19.836272,20.396550,0.560278,162.836084,355.365931
9,254.980080,38.743449,18.839137,17.997845,18.458894,18.229552,19.202247,0.114302,0.841292,0.380243,0.609585,-0.363110,-0.461050,-0.231707,-1.204402,0.229342,-0.743353,-0.972695,1,1,-1.824736,-2.010304,-0.235780,2.153346,2.057185,2.109884,2.083669,2.194850,18.545535,0.480830,17.997845,19.202247,1.204402,6.581243,257.906758


In [10]:
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

In [13]:
X = train_df
y = target_col

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y)

## Hyperparameter Search (Optuna)

In [14]:
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    model_name = trial.suggest_categorical('model', ['xgboost', 'catboost'])

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    if model_name == 'xgboost':
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
            'max_depth': trial.suggest_int('max_depth', 4, 12),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma': trial.suggest_float('gamma', 0.0, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 2.0),
            'reg_lambda': trial.suggest_float('reg_lambda', 0.5, 5.0),
            'random_state': RANDOM_STATE,
            'objective': 'multi:softprob',
            'num_class': 3,
            'eval_metric': 'mlogloss',
            'n_jobs': -1,
            'tree_method': 'hist',
        }
        model = XGBClassifier(**params)

    else:  # catboost
        params = {
            'iterations': trial.suggest_int('iterations', 200, 1000),
            'depth': trial.suggest_int('depth', 4, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
            'random_strength': trial.suggest_float('random_strength', 0.0, 2.0),
            'loss_function': 'MultiClass',
            'eval_metric': 'Accuracy',
            'random_seed': RANDOM_STATE,
            'verbose': 0,
            'thread_count': -1,
        }
        model = CatBoostClassifier(**params)

    scores = cross_val_score(model, X, y, cv=skf, scoring='balanced_accuracy', n_jobs=1)
    return scores.mean()

In [15]:
study = optuna.create_study(direction='maximize', study_name='stellar_classes')
study.optimize(objective, n_trials=50, show_progress_bar=True)

  0%|          | 0/50 [00:00<?, ?it/s]

[W 2026-06-01 18:26:49,588] Trial 7 failed with parameters: {'model': 'catboost', 'iterations': 926, 'depth': 7, 'learning_rate': 0.09796442992265275, 'l2_leaf_reg': 3.8804872251459104, 'bagging_temperature': 0.8829677697702252, 'random_strength': 0.7572225000405943} because of the following error: KeyboardInterrupt('').
Traceback (most recent call last):
  File "c:\Users\joao.p.mamede.dias\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\joao.p.mamede.dias\AppData\Local\Temp\ipykernel_29108\2576507742.py", line 47, in objective
    scores = cross_val_score(model, X, y, cv=skf, scoring='balanced_accuracy', n_jobs=1)
  File "c:\Users\joao.p.mamede.dias\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\_param_validation.py", line 218, in wrapper
    return func(*args, **kwargs)
  File "c:\Users\joao.p.mamede.dias\AppData\Local\Python\pythoncore-3.14-64\Lib

KeyboardInterrupt: 

In [ ]:
print(f"Best balanced_accuracy: {study.best_value:.4f}")
print(f"Best model: {study.best_params['model']}")
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
xgbc_model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    n_jobs=-1,
    tree_method='hist'
)

In [ ]:
xgbc_model.fit(X_train, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method

In [ ]:
y_pred = xgbc_model.predict(X_val)

In [ ]:
catboost_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function='MultiClass',
    eval_metric='Accuracy',
    random_seed=RANDOM_STATE,
    verbose=100,
    thread_count=-1
)

In [ ]:
catboost_model.fit(X_train, y_train, eval_set=(X_val, y_val))

## Results:

In [ ]:
print(balanced_accuracy_score(y_val, y_pred))

0.955754000495861


In [ ]:
y_pred_cat = catboost_model.predict(X_val).flatten()
print(balanced_accuracy_score(y_val, y_pred_cat))